In [1]:
import csv
import math
from pathlib import Path
from collections import defaultdict

DATA_DIR = Path("data")
ARTIFACT_DIR = Path("artifacts")

BASE_STATIONS = {
    0: (100, 100),
    1: (300, 100),
    2: (100, 300),
    3: (300, 300),
}

PREPARATION_DELAY = 2.0
RESERVATION_LIFETIME = 5.0
CONFIDENCE_THRESHOLD = 0.70
CAPACITY_PER_CELL = 10


def load_csv(path):
    with open(path, newline="") as f:
        return list(csv.DictReader(f))


def nearest_cell(x, y):
    return min(
        BASE_STATIONS,
        key=lambda cell: (
            (x - BASE_STATIONS[cell][0]) ** 2
            + (y - BASE_STATIONS[cell][1]) ** 2
        )
    )


trips = load_csv(DATA_DIR / "morning_test.csv")

predictions = load_csv(
    ARTIFACT_DIR / "pretrained_predictions.csv"
)

print("Trips loaded:", len(trips))
print("Predictions loaded:", len(predictions))
print("First trip:", trips[0])
print("First prediction:", predictions[0])

Trips loaded: 26024
Predictions loaded: 25324
First trip: {'trip_id': '0', 'time_s': '0.0', 'x_m': '54.61', 'y_m': '48.26', 'speed_m_s': '1.45', 'heading_rad': '0.884', 'serving_cell': '0'}
First prediction: {'trip_id': '0', 'time_s': '4.0', 'serving_cell': '0', 'target_time_s': '7.0', 'p_cell_0': '0.9999940395355225', 'p_cell_1': '2.903421545852325e-06', 'p_cell_2': '3.0617238735430874e-06', 'p_cell_3': '3.014060467876334e-08'}


In [2]:

import pandas as pd
import numpy as np
import math

# Load data
trips_df = pd.read_csv("data/morning_test.csv")
pred_df = pd.read_csv("artifacts/pretrained_predictions.csv")

# Cell configuration
BASE_STATIONS = {
    0: (100, 100),
    1: (300, 100),
    2: (100, 300),
    3: (300, 300)
}

CAPACITY = 10
PREPARATION_DELAY = 2.0
RESERVATION_LIFETIME = 5.0
CONFIDENCE_THRESHOLD = 0.70
PROJECTION_SECONDS = 3.0


def nearest_cell(x, y):
    return min(
        BASE_STATIONS,
        key=lambda c: (
            (x - BASE_STATIONS[c][0])**2
            + (y - BASE_STATIONS[c][1])**2
        )
    )


def predict_movement_cell(row):
    x = float(row.x_m)
    y = float(row.y_m)
    speed = float(row.speed_m_s)
    heading = float(row.heading_rad)

    future_x = x + speed * PROJECTION_SECONDS * math.cos(heading)
    future_y = y + speed * PROJECTION_SECONDS * math.sin(heading)

    return nearest_cell(future_x, future_y)


# Match predictions to trips using trip ID and time
pred_lookup = {
    (int(r.trip_id), float(r.time_s)): r
    for r in pred_df.itertuples(index=False)
}

print("Simulator configuration ready.")
print("Trip readings:", len(trips_df))
print("Prediction readings:", len(pred_lookup))


Simulator configuration ready.
Trip readings: 26024
Prediction readings: 25324


In [3]:

def run_simulation(strategy):
    """
    Strategies: reactive, movement, model
    All trips use one shared simulation clock and shared cell capacity.
    """

    reservations = {}
    previous_cells = {}

    delays = []
    handovers = 0
    correct_reservations = 0
    wrong_reservations = 0
    expired_reservations = 0
    unused_capacity_seconds = 0.0

    rows = trips_df.sort_values(
        ["time_s", "trip_id"]
    )

    for row in rows.itertuples(index=False):
        uid = int(row.trip_id)
        now = float(row.time_s)
        current_cell = int(row.serving_cell)

        # 1. Globally expire all reservations
        expired_users = [
            user_id
            for user_id, reservation in reservations.items()
            if now >= reservation["expiry"]
        ]

        for user_id in expired_users:
            reservation = reservations.pop(user_id)

            unused_capacity_seconds += (
                reservation["expiry"] - reservation["created"]
            )
            expired_reservations += 1

        # 2. Detect actual handover
        old_cell = previous_cells.get(uid)

        if old_cell is not None and old_cell != current_cell:
            handovers += 1

            reservation = reservations.get(uid)

            if reservation and reservation["cell"] == current_cell:
                delay = max(
                    0.0,
                    reservation["complete_at"] - now
                )
                correct_reservations += 1
                del reservations[uid]

            else:
                if reservation:
                    # Wrong reservation stays until expiry.
                    # It continues occupying cell capacity.
                    wrong_reservations += 1

                delay = PREPARATION_DELAY

            delays.append(delay)

        previous_cells[uid] = current_cell

        # 3. Reactive baseline does not reserve early
        if strategy == "reactive":
            continue

        # One pending reservation per user
        if uid in reservations:
            continue

        target_cell = None
        confidence = 0.0

        # 4. Select target cell
        if strategy == "movement":
            target_cell = predict_movement_cell(row)
            confidence = 1.0

        elif strategy == "model":
            prediction = pred_lookup.get((uid, now))

            if prediction is not None:
                probabilities = [
                    float(getattr(prediction, f"p_cell_{c}"))
                    for c in range(4)
                ]

                target_cell = int(np.argmax(probabilities))
                confidence = max(probabilities)

        else:
            raise ValueError(f"Unknown strategy: {strategy}")

        if target_cell is None or target_cell == current_cell:
            continue

        if confidence < CONFIDENCE_THRESHOLD:
            continue

        # 5. Check shared target-cell capacity
        occupied = sum(
            1
            for reservation in reservations.values()
            if reservation["cell"] == target_cell
        )

        if occupied >= CAPACITY:
            continue

        # 6. Create reservation
        reservations[uid] = {
            "cell": target_cell,
            "created": now,
            "complete_at": now + PREPARATION_DELAY,
            "expiry": now + RESERVATION_LIFETIME
        }

    # 7. Remaining reservations are unused at simulation end
    for reservation in reservations.values():
        unused_capacity_seconds += (
            reservation["expiry"] - reservation["created"]
        )

    return {
        "strategy": strategy,
        "handovers": handovers,
        "mean_delay": np.mean(delays) if delays else 0.0,
        "p95_delay": np.percentile(delays, 95) if delays else 0.0,
        "correct_early_preparation_rate": (
            correct_reservations / handovers if handovers else 0.0
        ),
        "correct_reservations": correct_reservations,
        "wrong_reservations": wrong_reservations,
        "expired_reservations": expired_reservations,
        "unused_capacity_seconds": unused_capacity_seconds
    }


# Re-run all strategies
results = []

for strategy in ["reactive", "movement", "model"]:
    results.append(run_simulation(strategy))

results_df = pd.DataFrame(results)
display(results_df.round(3))


,strategy,handovers,mean_delay,p95_delay,correct_early_preparation_rate,correct_reservations,wrong_reservations,expired_reservations,unused_capacity_seconds
0,reactive,176,2.000,2.0,0.000,0,0,0,0.0
1,movement,176,0.648,2.0,0.750,132,2,158,790.0
2,model,176,0.767,2.0,0.716,126,2,26,130.0


In [4]:

# Demo: correct reservation
now = 10.0

correct_reservation = {
    "cell": 1,
    "created": 8.0,
    "complete_at": 10.0,
    "expiry": 13.0
}

handover_time = 11.0

correct_delay = max(
    0,
    correct_reservation["complete_at"] - handover_time
)

print("CORRECT RESERVATION")
print("Reactive delay:", PREPARATION_DELAY, "seconds")
print("With reservation:", correct_delay, "seconds")

# Demo: wrong reservation
wrong_reservation = {
    "cell": 2,
    "created": 10.0,
    "expiry": 15.0
}

actual_handover_cell = 1

print("\nWRONG RESERVATION")
print("Reserved cell:", wrong_reservation["cell"])
print("Actual cell:", actual_handover_cell)
print("Reservation expires at:", wrong_reservation["expiry"])
print("Capacity occupied until expiry:", 
      wrong_reservation["expiry"] - wrong_reservation["created"],
      "seconds")
print("Reactive delay:", PREPARATION_DELAY, "seconds")


CORRECT RESERVATION
Reactive delay: 2.0 seconds
With reservation: 0 seconds

WRONG RESERVATION
Reserved cell: 2
Actual cell: 1
Reservation expires at: 15.0
Capacity occupied until expiry: 5.0 seconds
Reactive delay: 2.0 seconds
